# 00 — Окружение и данные

Ноутбук выполняется первым в новом клоне и проверяет, что окружение собрано правильно,
а датасет соревнования `alfa-bank-pd-credit-history` лежит на диске.

Порядок работы:

1. константы конфигурации: соревнование и каталог с сырыми данными
2. проверки окружения: интерпретатор, пакеты, импорты
3. проверка доступа к Kaggle API
4. скачивание датасета (zip ~390 MiB, 18 файлов) и распаковка
5. инвентаризация скачанного

Ноутбук проверяет окружение и загружает данные; анализа и обучения здесь нет.
Повторный запуск перекачивает архив и распаковывает его заново; учётные данные
Kaggle в репозиторий не записываются.

In [1]:
from __future__ import annotations

import csv
import importlib
import io
import re
import shutil
import subprocess
import sys
import zipfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    """Ищет корень репозитория по pyproject.toml: ноутбук работает из любого каталога внутри репозитория."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Не найден корень репозитория: нет pyproject.toml вверх по дереву")


REPO_ROOT = find_repo_root()


KAGGLE_COMPETITION = "alfa-bank-pd-credit-history"
RAW_DIR = REPO_ROOT / "data" / "raw"


def rel(path: Path) -> str:
    """Путь относительно корня репозитория: чтобы в выводах не печатался абсолютный путь."""
    try:
        return str(Path(path).resolve().relative_to(REPO_ROOT))
    except ValueError:
        return Path(path).name


print(f"корень:        {REPO_ROOT.name}")
print(f"соревнование:  {KAGGLE_COMPETITION}")
print(f"данные:        {rel(RAW_DIR)}")

корень:        kaggle-creditscore-alfa
соревнование:  alfa-bank-pd-credit-history
данные:        data/raw


In [2]:
FAILURES: list[str] = []


def report(label: str, passed: bool, detail: str = "", fix: str = "") -> bool:
    print(f"[{'PASS' if passed else 'FAIL'}] {label}" + (f" — {detail}" if detail else ""))
    if not passed:
        FAILURES.append(label)
        if fix:
            print(f"       исправить: {fix}")
    return passed


ANSI = re.compile(r"\x1b\[[0-9;]*[a-zA-Z]")


def strip_ansi(text: str) -> str:
    return ANSI.sub("", text)


def run(cmd: list[str], timeout: int = 600) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
        timeout=timeout,
    )


In [3]:
EXPECTED_VERSIONS = {
    "scikit-learn": "1.9.1",
    "pandas": "3.0.6",
    "numpy": "2.5.4",
    "pyarrow": "26.0.0",
    "openpyxl": "3.1.5",
}

report(
    "интерпретатор из .venv",
    Path(sys.prefix).name == ".venv",
    f"{sys.version.split()[0]}, prefix {rel(sys.prefix)}",
    "uv sync",
)

for name in (
    "kaggle",
    "matplotlib",
    "numpy",
    "pandas",
    "scikit-learn",
    "pyarrow",
    "openpyxl",
):
    try:
        installed = version(name)
        report(f"пакет {name}", True, installed)
        expected = EXPECTED_VERSIONS.get(name)
        if expected is not None and installed != expected:
            print(
                f"       [WARN] {name}: установлено {installed}, в README указано "
                f"{expected}; это предупреждение, не падение"
            )
    except PackageNotFoundError:
        report(f"пакет {name}", False, "не установлен", "uv sync")

for name in ("matplotlib", "numpy", "pandas", "sklearn", "pyarrow", "openpyxl"):
    try:
        importlib.import_module(name)
    except ImportError as exc:
        report(f"импорт {name}", False, str(exc), "uv sync")
    else:
        report(f"импорт {name}", True)


[PASS] интерпретатор из .venv — 3.12.7, prefix .venv
[PASS] пакет kaggle — 2.2.4
[PASS] пакет matplotlib — 3.11.2
[PASS] пакет numpy — 2.5.4
[PASS] пакет pandas — 3.0.6
[PASS] пакет scikit-learn — 1.9.1
[PASS] пакет pyarrow — 26.0.0
[PASS] пакет openpyxl — 3.1.5
[PASS] импорт matplotlib
[PASS] импорт numpy
[PASS] импорт pandas


[PASS] импорт sklearn
[PASS] импорт pyarrow
[PASS] импорт openpyxl


In [4]:
CREDENTIALS = Path.home() / ".kaggle" / "credentials.json"
report(
    "токен Kaggle на диске",
    CREDENTIALS.is_file(),
    "~/.kaggle/credentials.json",
    "uv run kaggle auth login",
)

listing = run(
    [
        sys.executable,
        "-m",
        "kaggle",
        "competitions",
        "files",
        "-c",
        KAGGLE_COMPETITION,
        "-v",
    ]
)
if listing.returncode == 0:
    rows = list(csv.DictReader(io.StringIO(strip_ansi(listing.stdout))))
    report("токен принят сервером Kaggle", True, f"файлов в соревновании: {len(rows)}")
else:
    report(
        "токен принят сервером Kaggle",
        False,
        strip_ansi((listing.stdout + listing.stderr).strip()[-500:]),
        "uv run kaggle auth login",
    )
    raise AssertionError(
        "Kaggle API недоступен. Проверь, что правила соревнования приняты, токен действует "
        "и сеть доступна: прими правила на kaggle.com и выполни `uv run kaggle auth login`. "
        "Пока команда не выполнится, данные не скачаются."
    )

if FAILURES:
    raise AssertionError(
        "Проваленные проверки до скачивания: " + ", ".join(FAILURES)
    )


[PASS] токен Kaggle на диске — ~/.kaggle/credentials.json


[PASS] токен принят сервером Kaggle — файлов в соревновании: 18


In [5]:
def clean_output(text: str) -> str:
    """Вырезает прогресс-бары kaggle: они идут в stderr символами возврата каретки."""
    chunks = strip_ansi(text).replace("\r", "\n").splitlines()
    return "\n".join(
        chunk.rstrip()
        for chunk in chunks
        if chunk.strip() and "B/s" not in chunk and not chunk.strip().endswith("%")
    )


DATA_DIR = RAW_DIR / "data_for_competition"
DOWNLOAD_DIR = RAW_DIR / ".download_tmp"
EXTRACT_DIR = RAW_DIR / ".extract_tmp"
OLD_DIR = RAW_DIR / ".data_for_competition_old"
ARCHIVE = DOWNLOAD_DIR / f"{KAGGLE_COMPETITION}.zip"
ATTEMPTS = 3
DOWNLOAD_CMD = [
    sys.executable,
    "-m",
    "kaggle",
    "competitions",
    "download",
    "-c",
    KAGGLE_COMPETITION,
    "-p",
    rel(DOWNLOAD_DIR),
    "-o",
]
EXPECTED_FILES = ("description.xlsx", "sample_submission.csv", "test_target.csv", "train_target.csv")


def validate_staging(staging_root: Path) -> None:
    """Проверяет staging до замены рабочего DATA_DIR."""
    train_parts = sorted((staging_root / "train_data").glob("*.pq"))
    test_parts = sorted((staging_root / "test_data").glob("*.pq"))
    assert len(train_parts) == 12, f"в staging {len(train_parts)} train-партиций, нужно 12"
    assert len(test_parts) == 2, f"в staging {len(test_parts)} test-партиций, нужно 2"
    missing = [name for name in EXPECTED_FILES if not (staging_root / name).is_file()]
    assert not missing, f"в staging нет файлов: {missing}"


def replace_data_dir(staging_root: Path) -> None:
    """Заменяет DATA_DIR только после успешной проверки staging; при ошибке восстанавливает старый."""
    shutil.rmtree(OLD_DIR, ignore_errors=True)
    had_old = DATA_DIR.exists()
    if had_old:
        DATA_DIR.rename(OLD_DIR)
    try:
        staging_root.rename(DATA_DIR)
    except Exception:
        if DATA_DIR.exists():
            shutil.rmtree(DATA_DIR, ignore_errors=True)
        if had_old and OLD_DIR.exists():
            OLD_DIR.rename(DATA_DIR)
        raise
    shutil.rmtree(OLD_DIR, ignore_errors=True)


def download_and_extract() -> None:
    """Качает архив и распаковывает его в staging; рабочий DATA_DIR меняется в конце."""
    shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    try:
        download = run(DOWNLOAD_CMD, timeout=1800)
        if download.returncode != 0:
            raise AssertionError(strip_ansi((download.stdout + download.stderr).strip()[-1000:]))
        print(clean_output(download.stdout))

        with zipfile.ZipFile(ARCHIVE) as archive:
            archive.extractall(EXTRACT_DIR)

        staging_root = EXTRACT_DIR / DATA_DIR.name
        if not staging_root.is_dir():
            staging_root = EXTRACT_DIR
        validate_staging(staging_root)

        # Старые данные не удаляются до успешной проверки staging.
        replace_data_dir(staging_root)
        ARCHIVE.unlink(missing_ok=True)
    except Exception:
        ARCHIVE.unlink(missing_ok=True)
        shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
        shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
        raise
    else:
        shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
        shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)


for attempt in range(1, ATTEMPTS + 1):
    try:
        download_and_extract()
    except (zipfile.BadZipFile, subprocess.TimeoutExpired) as exc:
        print(f"попытка {attempt}/{ATTEMPTS}: скачивание/архив не удались ({exc}), повторяем")
        if attempt == ATTEMPTS:
            raise AssertionError(f"датасет не получен после {ATTEMPTS} попыток: {exc}") from exc
    else:
        report("датасет скачан и распакован", True, rel(DATA_DIR))
        break


[PASS] датасет скачан и распакован — data/raw/data_for_competition


In [6]:
train_parts = sorted(DATA_DIR.glob("train_data/*.pq"))
test_parts = sorted(DATA_DIR.glob("test_data/*.pq"))
report("train_data_*.pq", len(train_parts) == 12, f"{len(train_parts)} шт. из 12")
report("test_data_*.pq", len(test_parts) == 2, f"{len(test_parts)} шт. из 2")
for name in ("description.xlsx", "sample_submission.csv", "test_target.csv", "train_target.csv"):
    report(name, (DATA_DIR / name).is_file(), rel(DATA_DIR / name))

[PASS] train_data_*.pq — 12 шт. из 12
[PASS] test_data_*.pq — 2 шт. из 2
[PASS] description.xlsx — data/raw/data_for_competition/description.xlsx
[PASS] sample_submission.csv — data/raw/data_for_competition/sample_submission.csv
[PASS] test_target.csv — data/raw/data_for_competition/test_target.csv
[PASS] train_target.csv — data/raw/data_for_competition/train_target.csv


In [7]:
files = sorted(path for path in DATA_DIR.rglob("*") if path.is_file())
inventory = pd.DataFrame(
    {
        "файл": [rel(path) for path in files],
        "MiB": [round(path.stat().st_size / 2**20, 2) for path in files],
    }
)
print(inventory.to_string(index=False))
print(f"\nфайлов: {len(files)}   всего: {inventory['MiB'].sum() / 1024:.2f} GiB")

                                                     файл   MiB
           data/raw/data_for_competition/description.xlsx  0.01
      data/raw/data_for_competition/sample_submission.csv 13.61
   data/raw/data_for_competition/test_data/test_data_0.pq 34.98
   data/raw/data_for_competition/test_data/test_data_1.pq 34.50
            data/raw/data_for_competition/test_target.csv  3.81
 data/raw/data_for_competition/train_data/train_data_0.pq 29.31
 data/raw/data_for_competition/train_data/train_data_1.pq 31.09
data/raw/data_for_competition/train_data/train_data_10.pq 33.71
data/raw/data_for_competition/train_data/train_data_11.pq 35.75
 data/raw/data_for_competition/train_data/train_data_2.pq 30.69
 data/raw/data_for_competition/train_data/train_data_3.pq 31.16
 data/raw/data_for_competition/train_data/train_data_4.pq 30.49
 data/raw/data_for_competition/train_data/train_data_5.pq 31.60
 data/raw/data_for_competition/train_data/train_data_6.pq 31.90
 data/raw/data_for_competition/train_dat

In [8]:
if FAILURES:
    raise AssertionError("Проваленные проверки: " + ", ".join(FAILURES))

print(
    "Окружение готово, данные на месте. "
    "Следующий шаг — разведочный анализ в notebooks/01-eda.ipynb."
)

Окружение готово, данные на месте. Следующий шаг — разведочный анализ в notebooks/01-eda.ipynb.


## Замечания

- Гейт `if FAILURES:` стоит в ячейке 4 **до** скачивания: при провале окружения
  или Kaggle API ноутбук останавливается, не начиная загрузку.
- `kaggle competitions files ... -v` печатает CSV; список разбирается через
  `csv.DictReader`, поэтому число файлов — это `len(rows)`, а не `len(rows) - 2`.
- Скачивание идёт в `data/raw/.download_tmp`, распаковка — в `.extract_tmp`.
  Рабочий `data_for_competition` заменяется только после проверки 12 train,
  2 test и обязательных файлов; при ошибке старый каталог остаётся на месте.
- Старые данные не удаляются до успешной проверки staging. Замена каталога
  делается переименованием, старый каталог удаляется только после успеха.
- У скачивания отдельный `timeout=1800` секунд: архив ~390 MiB не должен
  обрываться на десятиминутном лимите. Архив удаляется после успешной распаковки или в обработчике ошибки.
- В инвентаризации используются `MiB` и `GiB` (двоичные единицы), а не десятичные мегабайты и гигабайты.
- **Ноутбук перезапущен 2026-10-22.** Выводы выше относятся к текущей версии
  кода; полный прогон включает скачивание ~390 MiB и распаковку датасета.
